<a href="https://colab.research.google.com/github/BryanHinostroza/lab07-bh/blob/develop/LABORATORIO_7_REGRESI%C3%93N_LOG%C3%8DSTICA_M%C3%81QUINAS_DE_VECTOR_DE_SOPORTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importación de librerías y definición de variables globales

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.svm import SVC
import statsmodels.api as sm

# URL del dataset
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

# Nombres de columnas como indica el dataset original
COLUMN_NAMES = ['id', 'clump_thickness', 'uniformity_cell_size', 'uniformity_cell_shape',
                'marginal_adhesion', 'single_epithelial_cell_size', 'bare_nuclei',
                'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class']


# Función para cargar los datos

In [4]:
def get_data_from_url(url):
    """
    Retrieve data from a csv into a dataframe.
    input:
    - url: URL of the csv file
    output:
    - dataframe
    """
    return pd.read_csv(url, names=COLUMN_NAMES)


# Limpieza del dataset

In [5]:
def clean_data(df):
    """
    Limpia el dataset eliminando valores nulos o incorrectos.
    """
    df = df.replace('?', np.nan)
    df = df.dropna()
    df['bare_nuclei'] = df['bare_nuclei'].astype(int)
    df['class'] = df['class'].apply(lambda x: 1 if x == 4 else 0)  # Maligno = 1, Benigno = 0
    df = df.drop(columns=['id'])  # Eliminamos la columna de ID
    return df


# Cálculo de Information Value (IV)

In [6]:
def calculate_iv(df, target):
    """
    Calcula el Information Value (IV) para cada variable independiente.
    """
    def woe_iv(df, feature, target):
        lst = []
        for val in np.sort(df[feature].unique()):
            count_event = ((df[feature] == val) & (df[target] == 1)).sum()
            count_non_event = ((df[feature] == val) & (df[target] == 0)).sum()
            lst.append([val, count_event, count_non_event])
        data = pd.DataFrame(lst, columns=['Value', 'Event', 'NonEvent'])
        data['Event'] = data['Event'].astype('float')
        data['NonEvent'] = data['NonEvent'].astype('float')
        data['Dist_Event'] = data['Event'] / data['Event'].sum()
        data['Dist_NonEvent'] = data['NonEvent'] / data['NonEvent'].sum()
        data['WOE'] = np.log(data['Dist_Event'] / data['Dist_NonEvent'])
        data['IV'] = (data['Dist_Event'] - data['Dist_NonEvent']) * data['WOE']
        iv = data['IV'].sum()
        return iv

    iv_dict = {}
    for col in df.columns:
        if col != target:
            iv = woe_iv(df, col, target)
            iv_dict[col] = iv
    return pd.Series(iv_dict).sort_values(ascending=False)


# Selección de variables predictoras fuertes

In [7]:
def select_strong_predictors(iv_series, threshold=0.02):
    """
    Selecciona variables con IV significativo (por encima del umbral).
    """
    return iv_series[iv_series >= threshold].index.tolist()


# Carga, limpieza y selección de variables

In [8]:
# Cargar y limpiar datos
df = get_data_from_url(DATA_URL)
df = clean_data(df)

# Calcular IV
iv_series = calculate_iv(df, 'class')
print("Information Value por variable:\n", iv_series)

# Seleccionar variables con buen poder predictivo
SELECTED_FEATURES = select_strong_predictors(iv_series)


Information Value por variable:
 clump_thickness                inf
uniformity_cell_size           inf
uniformity_cell_shape          inf
marginal_adhesion              inf
single_epithelial_cell_size    inf
bare_nuclei                    inf
bland_chromatin                inf
normal_nucleoli                inf
mitoses                        inf
dtype: float64


# Separación de datos en entrenamiento y prueba

In [9]:
# Separar datos de entrenamiento y prueba
X = df[SELECTED_FEATURES]
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


# Modelo de regresión logística (significancia estadística + métricas)

In [10]:
# Evaluar significancia estadística con statsmodels
X_train_sm = sm.add_constant(X_train)
logit_model = sm.Logit(y_train, X_train_sm).fit()
print(logit_model.summary())


Optimization terminated successfully.
         Current function value: 0.073760
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                  class   No. Observations:                  512
Model:                          Logit   Df Residuals:                      502
Method:                           MLE   Df Model:                            9
Date:                Thu, 01 May 2025   Pseudo R-squ.:                  0.8842
Time:                        00:31:16   Log-Likelihood:                -37.765
converged:                       True   LL-Null:                       -326.13
Covariance Type:            nonrobust   LLR p-value:                2.070e-118
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const                         -10.0640      1.408     -7.150      0.000   

# Modelo de regresión logística con sklearn

In [11]:
# Modelo con sklearn
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Métricas del modelo logístico
print("Precisión Regresión Logística:", metrics.accuracy_score(y_test, y_pred_lr))
print("Reporte de clasificación:\n", metrics.classification_report(y_test, y_pred_lr))


Precisión Regresión Logística: 0.9532163742690059
Reporte de clasificación:
               precision    recall  f1-score   support

           0       0.94      0.99      0.96       103
           1       0.98      0.90      0.94        68

    accuracy                           0.95       171
   macro avg       0.96      0.94      0.95       171
weighted avg       0.95      0.95      0.95       171



# Modelo SVM y métricas

In [12]:
svm = SVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

# Métricas del modelo SVM
print("Precisión SVM:", metrics.accuracy_score(y_test, y_pred_svm))
print("Reporte de clasificación SVM:\n", metrics.classification_report(y_test, y_pred_svm))


Precisión SVM: 0.9473684210526315
Reporte de clasificación SVM:
               precision    recall  f1-score   support

           0       0.94      0.98      0.96       103
           1       0.97      0.90      0.93        68

    accuracy                           0.95       171
   macro avg       0.95      0.94      0.94       171
weighted avg       0.95      0.95      0.95       171



# Comparación de modelos

In [13]:
print("Comparación de modelos:")
print(f"Logistic Regression Accuracy: {metrics.accuracy_score(y_test, y_pred_lr):.4f}")
print(f"SVM Accuracy: {metrics.accuracy_score(y_test, y_pred_svm):.4f}")


Comparación de modelos:
Logistic Regression Accuracy: 0.9532
SVM Accuracy: 0.9474
